# Jour 2 -- Data manipulation (groupby, merge, dates, texte)

**Périmètre strict de cette session :** `groupby`, `agg`, `merge`, `join`,
`concat`, `datetime` (`pd.to_datetime`, `.dt`), `.str`, `apply`, `map`,
`np.where`. On suppose les fondamentaux du Jour 1 acquis (`loc`/`iloc`,
`isna`/`fillna`, `value_counts`, etc.) -- ils seront réutilisés sans être
ré-expliqués.

**Objectif chrono : 15 minutes pour tout le notebook.** C'est plus long
que le Jour 1 car les opérations s'enchaînent (ex. nettoyer une colonne
texte avant de merger dessus). Si un exercice dépasse 2 minutes, notez-le
et repassez dessus après la solution.

## Comment travailler ce notebook
1. Lancez un chrono de 15 minutes.
2. Chaque exercice suit le format réel CodeSignal : **Description**
   (contexte métier), **Instruction** (tâches numérotées, types de
   colonnes précisés entre parenthèses) et **Output Requirements** (ce
   qui sera vérifié) -- pas d'indice, pas d'aide.
3. Écrivez le code dans la cellule `# VOTRE CODE ICI`.
4. À la fin (ou quand le chrono sonne), ouvrez
   `../solutions/day2_data_manipulation_solutions.ipynb` -- chaque
   solution explique le *pourquoi*, pas seulement le *comment*.
5. Relisez aussi le PDF `../pdf/jour2_data_manipulation.pdf` : il détaille
   le raisonnement à avoir pour chaque famille de question, dans l'esprit
   BCG X.


In [ ]:
# ---------------------------------------------------------------------------
# Run this cell first. Paths are relative to this notebook's folder.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

DATA = "../data"
drivers = pd.read_csv(f"{DATA}/drivers.csv")
trips = pd.read_csv(f"{DATA}/trips.csv")
print(drivers.shape, trips.shape)


---
## Exercice 1 -- `pd.to_datetime`, `.dt`  *(~2 min)*

### Description
`trips["trip_date"]` est chargée en texte (`object`) depuis le CSV. Avant
tout calcul temporel, il faut la convertir en vrai type date.

### Instruction
1. Convertissez `trips["trip_date"]` en `datetime64` avec `pd.to_datetime`
   -- réaffectez dans la même colonne.
2. Créez une colonne `trip_month` (entier 1-12) et une colonne
   `trip_weekday` (nom du jour en anglais, ex. `"Monday"`) à partir de
   `trip_date`.
3. Comptez le nombre de courses par mois (`trip_month`).

### Output Requirements
`trips["trip_date"]` de dtype `datetime64[ns]`, les colonnes `trip_month`
et `trip_weekday` ajoutées à `trips`, et une Series du nombre de courses
par mois.


In [ ]:
# VOTRE CODE ICI



---
## Exercice 2 -- Nettoyage `.str`  *(~2 min)*

### Description
`trips["dropoff_city"]` (type `str`) contient des espaces parasites et une
casse incohérente (`" paris"`, `"LYON "`, `"Nice"`...). Il faut la
normaliser avant de pouvoir la comparer à `pickup_city`.

### Instruction
1. Créez `dropoff_city_clean` : `dropoff_city` sans espaces en début/fin
   (`strip`) et avec une casse homogène de type `"Paris"` (`title`).
2. À partir de `comment` (type `str`, contient des `NaN`), créez une
   colonne booléenne `mentions_client` qui vaut `True` si le commentaire
   contient la sous-chaîne `"client"` **sans tenir compte de la casse**
   (les `NaN` doivent donner `False`, pas planter).
3. Comptez combien de lignes ont `pickup_city == dropoff_city_clean`
   (aller-retour dans la même ville).

### Output Requirements
`dropoff_city_clean`, `mentions_client` (dtype `bool`, aucun `NaN`), et un
entier (le compte de trajets dans la même ville).

**Note.** `.str.contains(..., na=False)` évite que les `NaN` fassent
planter le masque booléen en aval.


In [ ]:
# VOTRE CODE ICI



---
## Exercice 3 -- `groupby` + `agg` (une métrique)  *(~2 min)*

### Description
La direction veut une première vue de la performance par ville de départ
(`pickup_city`).

### Instruction
1. Calculez la `fare_eur` moyenne par `pickup_city` -- stockez dans
   `avg_fare_by_city`, trié par valeur décroissante.
2. Calculez, pour chaque `pickup_city`, le **nombre** de courses -- stockez
   dans `trip_count_by_city`.
3. Combinez les deux dans un seul DataFrame `city_summary` avec deux
   colonnes nommées `avg_fare` et `n_trips`.

### Output Requirements
`avg_fare_by_city`, `trip_count_by_city`, et `city_summary` (index =
`pickup_city`, colonnes `avg_fare` et `n_trips`).


In [ ]:
# VOTRE CODE ICI



---
## Exercice 4 -- `groupby` + `agg` (plusieurs métriques)  *(~3 min)*

### Description
Le besoin se précise : il faut plusieurs statistiques par ville **et** par
statut de course en une seule opération, avec des noms de colonnes clairs.

### Instruction
1. Avec `.agg(...)` (syntaxe *named aggregation*), calculez par
   `pickup_city` : `total_revenue` (somme de `fare_eur`), `avg_distance`
   (moyenne de `distance_km`), `n_trips` (nombre de lignes) -- stockez
   dans `city_stats`.
2. Groupez par `pickup_city` **et** `status`, et calculez la `fare_eur`
   moyenne pour chaque combinaison -- stockez dans `city_status_avg`
   (résultat multi-index).
3. À partir de `city_status_avg`, remettez `status` en colonnes avec
   `.unstack()` -- stockez dans `city_status_wide`.

### Output Requirements
`city_stats` (colonnes `total_revenue`, `avg_distance`, `n_trips`),
`city_status_avg` (Series ou DataFrame multi-index), `city_status_wide`
(une colonne par valeur de `status`).

**Note.** La *named aggregation* s'écrit
`df.groupby("col").agg(nom_sortie=("col_source", "fonction"))` -- c'est le
moyen le plus lisible d'éviter les `MultiIndex` de colonnes imbriquées.


In [ ]:
# VOTRE CODE ICI



---
## Exercice 5 -- `merge` (inner vs left)  *(~3 min)*

### Description
Il faut enrichir chaque course avec les informations du chauffeur qui l'a
effectuée (`age`, `rating`, `city` du chauffeur). Certains `driver_id` de
`trips` n'existent pas dans `drivers` (chauffeurs mal renseignés/tests).

### Instruction
1. Faites un `merge` **inner** entre `trips` et `drivers` sur
   `driver_id` -- stockez dans `trips_inner`. Comparez son nombre de
   lignes à celui de `trips`.
2. Faites un `merge` **left** (base = `trips`) sur `driver_id` --
   stockez dans `trips_left`. Le nombre de lignes doit être identique à
   `trips`.
3. Dans `trips_left`, isolez les courses dont le chauffeur est introuvable
   dans `drivers` (colonne `age`, ou toute colonne venant de `drivers`,
   est `NaN` après le merge) -- stockez dans `orphan_trips`.

### Output Requirements
`trips_inner` (moins de lignes que `trips`), `trips_left` (même nombre de
lignes que `trips`), `orphan_trips` (non vide).

**Note.** `suffixes=("_trip", "_driver")` est utile si `merge` détecte des
noms de colonnes en conflit (ici `status` existe potentiellement des deux
côtés selon vos étapes précédentes -- vérifiez `trips_left.columns` après
coup).


In [ ]:
# VOTRE CODE ICI



---
## Exercice 6 -- `join` (sur l'index)  *(~2 min)*

### Description
La méthode `.join()` est la variante de `merge` pensée pour joindre sur
l'**index** plutôt que sur une colonne -- utile quand un DataFrame est
déjà indexé par la clé de jointure (typiquement après un `groupby` ou un
`set_index`).

### Instruction
1. Construisez `drivers_by_id`, une copie de `drivers` indexée par
   `driver_id` (`set_index`).
2. Construisez `trips_by_driver`, une copie de `trips` indexée par
   `driver_id`.
3. Avec `.join()` (pas `merge`), ajoutez la colonne `rating` de
   `drivers_by_id` à `trips_by_driver` -- stockez dans `joined`. Utilisez
   `how="left"`.

### Output Requirements
`drivers_by_id` et `trips_by_driver` indexés par `driver_id`, `joined`
avec le même nombre de lignes que `trips` et une colonne `rating`
supplémentaire.

**Note.** `df_a.join(df_b[["col"]])` exige que `df_a` et `df_b` partagent
le même type d'index (ici `driver_id`) -- c'est la contrainte qui
distingue `.join()` de `merge`, plus flexible mais plus verbeux.


In [ ]:
# VOTRE CODE ICI



---
## Exercice 7 -- `concat`  *(~2 min)*

### Description
Les données de janvier-février et de mars sont parfois livrées dans des
fichiers séparés puis rassemblées. On simule ce cas ici pour pratiquer
`concat` (empilement de lignes) et son usage pour ajouter une colonne
d'origine.

### Instruction
1. Découpez `trips` (déjà convertie en dates à l'exercice 1) en
   `trips_q1a` (mois 1 et 2) et `trips_q1b` (mois 3), selon
   `trip_month`.
2. Recollez les deux avec `pd.concat`, `ignore_index=True` -- stockez dans
   `trips_recombined`. Vérifiez que sa forme est identique à `trips`.
3. Recommencez le `concat` avec l'option `keys=["jan_fev", "mars"]` --
   stockez dans `trips_with_source`, puis affichez son index (il doit
   être un `MultiIndex` avec le nom du lot en premier niveau).

### Output Requirements
`trips_q1a`, `trips_q1b`, `trips_recombined` (même `shape` que `trips`),
`trips_with_source` (MultiIndex à deux niveaux).

**Note.** `pd.concat` empile par défaut selon les **labels de colonnes**
communs -- si les deux DataFrames n'ont pas exactement les mêmes colonnes,
des `NaN` apparaissent là où une colonne manque d'un côté. Ici les
colonnes sont identiques, donc pas de piège de ce type.


In [ ]:
# VOTRE CODE ICI



---
## Exercice 8 -- `apply` sur une colonne  *(~2 min)*

### Description
La finance veut une catégorie de course selon la distance, avec une règle
qui ne se laisse pas exprimer proprement avec un simple `map`.

### Instruction
1. Écrivez une fonction `distance_bucket(km)` qui renvoie `"short"` si
   `km < 5`, `"medium"` si `5 <= km < 15`, `"long"` sinon (gérez aussi le
   cas `NaN` -> renvoyer `"unknown"`).
2. Appliquez-la à `trips["distance_km"]` avec `.apply()` -- stockez le
   résultat dans la nouvelle colonne `trips["distance_bucket"]`.
3. Avec `.apply(axis=1)` sur `trips`, créez `trips["price_per_km"]` =
   `fare_eur / distance_km`, en renvoyant `np.nan` si `distance_km` vaut 0
   ou est manquant (pour éviter une division par zéro).

### Output Requirements
`trips["distance_bucket"]` (valeurs parmi `short`/`medium`/`long`/
`unknown`, aucun `NaN`), `trips["price_per_km"]` (float, `NaN` géré
explicitement, jamais `inf`).

**Note.** `.apply(axis=1)` est nettement plus lent que les opérations
vectorisées -- utile pour une logique ligne par ligne complexe, mais à
éviter en boucle sur de gros volumes si une formulation vectorisée existe.


In [ ]:
# VOTRE CODE ICI



---
## Exercice 9 -- `map`  *(~1 min)*

### Description
Le service commercial fournit une grille tarifaire de zone par ville, à
appliquer telle quelle -- un cas d'école pour `map` (correspondance simple
valeur -> valeur via un dict), à ne pas confondre avec `apply`.

### Instruction
1. À partir du dict `zone_map = {"Paris": "Zone A", "Lyon": "Zone B",
   "Marseille": "Zone B", "Nice": "Zone C"}`, créez
   `trips["pickup_zone"]` en mappant `pickup_city` (`.map`).
2. Les valeurs de `pickup_city` absentes du dict doivent donner `NaN` --
   vérifiez-le et comptez combien de lignes sont dans ce cas.

### Output Requirements
`trips["pickup_zone"]`, et un entier (nombre de villes hors grille
tarifaire, normalement 0 ici puisque les 4 villes sont couvertes -- codez
la vérification comme si ce n'était pas garanti).

**Note.** `map` attend un dict ou une Series ; pour une logique
conditionnelle (bornes, `if/elif`), c'est `apply` qu'il faut, pas `map`.


In [ ]:
# VOTRE CODE ICI



---
## Exercice 10 -- `np.where`  *(~2 min)*

### Description
On veut une colonne de flag rapide, sans écrire de fonction ni utiliser
`apply` : `np.where` est le réflexe vectorisé pour un `if/else` simple sur
une Series entière.

### Instruction
1. Créez `trips["is_long_trip"]` : `True` si `distance_km > 15`, `False`
   sinon (y compris quand `distance_km` est manquant) avec `np.where`.
2. Créez `trips["fare_flag"]` avec trois valeurs possibles selon
   `fare_eur` : `"low"` si `< 15`, `"high"` si `> 40`, `"normal"` sinon --
   en **imbriquant deux appels** à `np.where` (pas de fonction, pas
   d'`apply`).

### Output Requirements
`trips["is_long_trip"]` (dtype `bool`), `trips["fare_flag"]` (valeurs
parmi `low`/`normal`/`high`).

**Note.** `np.where(cond, val_si_vrai, val_si_faux)` est vectorisé --
largement plus rapide qu'`apply` pour ce genre de règle, et c'est ce que
les recruteurs attendent dès qu'une condition simple porte sur une colonne
entière.


In [ ]:
# VOTRE CODE ICI -- chrono : vous devriez avoir fini le notebook maintenant.



---
# CE QUE VOUS DEVEZ SAVOIR TAPER DE MÉMOIRE

```
pd.to_datetime(df["col"])
df["col"].dt.month / .dt.day_name() / .dt.year

df["col"].str.strip().str.title()
df["col"].str.contains("x", case=False, na=False)

df.groupby("key")["val"].mean() / .sum() / .count()
df.groupby("key").agg(nom=("col", "fonction"))
df.groupby(["k1", "k2"])["val"].mean().unstack()

df_a.merge(df_b, on="key", how="inner"/"left"/"right"/"outer")
df_a.set_index("key").join(df_b.set_index("key"))

pd.concat([df1, df2], ignore_index=True)
pd.concat([df1, df2], keys=["a", "b"])

df["col"].apply(fonction)              # Series -> Series, logique complexe
df.apply(fonction, axis=1)             # ligne -> valeur, logique multi-colonnes
df["col"].map(dict_ou_series)          # correspondance simple valeur->valeur

np.where(condition, val_si_vrai, val_si_faux)
```

**Règle d'or :** `map` pour une correspondance simple (dict), `apply` pour
une logique arbitraire (fonction, conditions multiples), `np.where` pour
un `if/else` vectorisé sans fonction. Pour joindre deux tables : `merge`
par défaut, `join` seulement si les deux DataFrames sont déjà indexés par
la clé commune.
